# Candidate Recommendation — TF-IDF Improved (Employer view)
Query: **Job description → Top-K candidates**

Pipeline:
1) Retrieve Top-N by TF-IDF cosine (semantic_score)
2) Enrich with heuristics: taxonomy, seniority, skills
3) Re-rank with weighted final_score

Outputs:
- `./outputs/candrec_tfidf/improved_topk.csv`
- `./outputs/candrec_tfidf/metrics_rows.csv`
- `./outputs/candrec_tfidf/metrics_summary.csv`


In [1]:
import os, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
from scipy import sparse

DATA_DIR = "./datasets"
OUT_DIR = "./outputs/candrec_tfidf"
os.makedirs(OUT_DIR, exist_ok=True)

JOBS_PATH = os.path.join(DATA_DIR, "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx")
RESUMES_PATH = os.path.join(DATA_DIR, "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")



jobs = pd.read_excel(JOBS_PATH)
resumes = pd.read_csv(RESUMES_PATH)

print("jobs shape:", jobs.shape)
print("sources:", jobs["source"].value_counts().head())

def clean_text(s: str) -> str:
    s = str(s or "")
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"[^a-z0-9\+\#\.\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# cached TF-IDF on resumes
VEC_PATH = os.path.join(OUT_DIR, "tfidf_vectorizer.joblib")
X_PATH = os.path.join(OUT_DIR, "tfidf_resumes.npz")
resume_texts = resumes["Resume"].fillna("").apply(clean_text)

if os.path.exists(VEC_PATH) and os.path.exists(X_PATH):
    vectorizer = joblib.load(VEC_PATH)
    X_resumes = sparse.load_npz(X_PATH)
else:
    vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=120_000)
    X_resumes = vectorizer.fit_transform(resume_texts)
    joblib.dump(vectorizer, VEC_PATH)
    sparse.save_npz(X_PATH, X_resumes)

print("TF-IDF:", X_resumes.shape)

sample = jobs.loc[0, "description"]  # hoặc "cleaned_description" tuỳ file


jobs shape: (89277, 5)
sources: source
swe    33629
it     21953
cs     13291
pm     12222
ds      8182
Name: count, dtype: int64
TF-IDF: (962, 44274)


## Heuristics: taxonomy, seniority, skills


In [2]:
# --- taxonomy labels for IT taxonomy (lightweight heuristic) ---
TAX_LABELS = ["backend","frontend","data","devops","mobile","security","qa","pm","other"]

def infer_taxonomy(text: str) -> str:
    t = clean_text(text)
    # order matters
    rules = [
        ("data", r"\b(data|ml|machine learning|deep learning|ai|analytics|etl|bi|sql|spark|hadoop|pytorch|tensorflow)\b"),
        ("devops", r"\b(devops|docker|kubernetes|k8s|ci\/?cd|jenkins|terraform|ansible|aws|gcp|azure|cloud)\b"),
        ("security", r"\b(security|soc|siem|pentest|penetration|forensic|iso 27001|iam)\b"),
        ("mobile", r"\b(android|ios|flutter|react native|swift|kotlin)\b"),
        ("frontend", r"\b(frontend|front-end|react|angular|vue|html|css|javascript|typescript)\b"),
        ("backend", r"\b(backend|back-end|api|microservice|spring|django|flask|node|express|nest|php|laravel|dotnet|\.net)\b"),
        ("qa", r"\b(qa|tester|testing|selenium|cypress|playwright|automation test)\b"),
        ("pm", r"\b(product manager|project manager|scrum|agile|business analyst|ba)\b"),
    ]
    for lab, pat in rules:
        if re.search(pat, t):
            return lab
    return "other"

def infer_seniority(text: str) -> str:
    t = clean_text(text)
    if re.search(r"\b(intern|internship|trainee)\b", t): return "intern"
    if re.search(r"\b(junior|jr\b|entry level|fresher)\b", t): return "junior"
    if re.search(r"\b(senior|sr\b|lead|principal|staff)\b", t): return "senior"
    if re.search(r"\b(manager|head)\b", t): return "manager"
    return "mid"

# --- skills list (keep short but meaningful) ---
# --- skills list (keep short but meaningful) ---
SKILLS = [
    "python","java","javascript","typescript","c++","c#",".net","rust","php",
    "react","angular","vue","node","express","nest","spring","django","flask",
    "sql","mysql","postgres","mongodb","redis",
    "docker","kubernetes","aws","gcp","azure","terraform","jenkins",
    "pandas","numpy","scikit-learn","tensorflow","pytorch","spark",
    "tableau","power bi",
    # NOTE: remove "go" from here to avoid false positive
]

SUPPORT_SKILLS = [
  "helpdesk","service desk","it support","technical support","desktop support","remote support",
  "troubleshooting","ticketing","incident management",
  "windows","linux","macos","active directory","azure ad","group policy",
  "office 365","microsoft 365","exchange","outlook","teams",
  "vpn","voip","tcp/ip","dns","dhcp","wifi","lan","wan",
  "hardware","printer","printers","laptop","desktop",
  "intune","sccm","jamf"
]

ALL_SKILLS = SKILLS + SUPPORT_SKILLS

_skill_pat = re.compile(r"\b(" + "|".join([re.escape(s) for s in ALL_SKILLS]) + r")\b", re.I)

# go/golang pattern (strict)
_go_pat = re.compile(r"\b(golang|go\s+language|go-lang)\b", re.I)

def extract_skills(text: str):
    t = clean_text(text)

    found = set(m.group(1).lower() for m in _skill_pat.finditer(t))

    # normalize some variants
    normed = set()
    for x in found:
        if x in [".net", "net"]:
            normed.add("dotnet")
        elif x == "postgres":
            normed.add("postgresql")   # optional
        else:
            normed.add(x)

    # add go only when it truly means Golang
    if _go_pat.search(t):
        normed.add("go")

    return sorted(normed)

print("HAS 'go' as word:", bool(re.search(r"\bgo\b", clean_text(sample))))
print("HAS golang:", bool(re.search(r"\bgolang\b", clean_text(sample))))



HAS 'go' as word: True
HAS golang: False


In [3]:
# --- preprocess resumes for taxonomy/seniority/skills ---
# Make sure these columns exist BEFORE calling recommend_candidates_*()
resumes["clean_resume"] = resumes["Resume"].fillna("").apply(clean_text)

resumes["cv_tax"] = resumes["clean_resume"].apply(infer_taxonomy)
resumes["cv_sen"] = resumes["clean_resume"].apply(infer_seniority)
resumes["cv_skills"] = resumes["clean_resume"].apply(extract_skills)

print("resume columns:", [c for c in ["cv_tax","cv_sen","cv_skills"] if c in resumes.columns])
print(resumes[["Category","cv_tax","cv_sen","cv_skills"]].head())


resume columns: ['cv_tax', 'cv_sen', 'cv_skills']
       Category cv_tax   cv_sen  \
0  Data Science   data  manager   
1  Data Science   data      mid   
2  Data Science   data   senior   
3  Data Science   data   senior   
4  Data Science   data   intern   

                                           cv_skills  
0  [angular, docker, flask, java, javascript, mys...  
1                                      [aws, python]  
2  [django, flask, java, linux, mysql, python, sq...  
3                    [python, sql, tableau, windows]  
4                                     [java, python]  


## Retrieval + Re-ranking


In [5]:
def build_job_query(job_row: pd.Series) -> str:
    title = str(job_row.get("title", "") or "")
    desc = str(job_row.get("cleaned_description", job_row.get("description", "")) or "")
    return clean_text(title + " " + desc)

STOP_SKILLS = {"teams", "communication", "teamwork", "leadership", "problem solving"}

def rerank(job_text: str, job_tax: str, job_sen: str, job_skills: list,
           cand_idx: np.ndarray, semantic_scores: np.ndarray):

    rows = []

    # 1. Loại skill quá chung
    STOP_SKILLS = {
        "teams", "communication", "teamwork",
        "leadership", "problem solving", "management"
    }

    js = [s.lower() for s in job_skills if s.lower() not in STOP_SKILLS]
    js_set = set(js)

    # 2. Hàm soft skill matching (substring)
    def soft_match(job_skills, cv_skills):
        matched = []
        for j in job_skills:
            for c in cv_skills:
                if j in c or c in j:
                    matched.append(j)
                    break
        return list(set(matched))

    for ri, sem in zip(cand_idx, semantic_scores):
        cv = resumes.iloc[int(ri)]
        cv_sk = [s.lower() for s in cv["cv_skills"] if s.lower() not in STOP_SKILLS]
        cv_set = set(cv_sk)

        # 3. Skill matching
        matched = soft_match(js, cv_sk)

        if len(js_set) > 0:
            overlap = len(matched) / len(js_set)
        else:
            overlap = 0.0

        # 4. Taxonomy match
        tax_match = 1.0 if cv["cv_tax"] == job_tax else 0.0

        # 5. Seniority penalty (đã fix mạnh)
        mismatch = 0.0
        if job_sen in ["senior", "manager"] and cv["cv_sen"] in ["intern", "junior"]:
            mismatch = 1.0
        elif job_sen in ["intern", "junior"] and cv["cv_sen"] in ["senior", "manager"]:
            mismatch = 1.0

        # 6. Final score
        final = (
            0.70 * float(sem)
            + 0.20 * overlap
            + 0.10 * tax_match
            - 0.15 * mismatch
        )
        final = max(0.0, min(1.0, final))

        # 7. Nếu không có skill match thì hiển thị rõ
        if len(matched) == 0:
            matched_display = ["(no explicit skill match)"]
        else:
            matched_display = matched[:8]

        rows.append({
            "resume_index": int(ri),
            "Category": cv.get("Category", ""),
            "semantic_score": float(sem),
            "final_score": final,
            "cv_tax": cv["cv_tax"],
            "cv_sen": cv["cv_sen"],
            "matched_skills": matched_display,
            "skill_overlap": overlap,
            "tax_match": tax_match,
            "seniority_mismatch": mismatch,
        })

    df = (
        pd.DataFrame(rows)
        .sort_values("final_score", ascending=False)
        .reset_index(drop=True)
    )

    return df

def recommend_candidates_improved(job_index: int, top_k: int = 10, retrieve_n: int = 200):
    job_row = jobs.iloc[job_index]
    job_text = build_job_query(job_row)
    job_tax = infer_taxonomy(job_text)
    job_sen = infer_seniority(job_text)
    job_skills = extract_skills(job_text)

    q_vec = vectorizer.transform([job_text])
    sims = cosine_similarity(q_vec, X_resumes).ravel()
    cand_idx = np.argsort(-sims)[:retrieve_n]
    df = rerank(job_text, job_tax, job_sen, job_skills, cand_idx, sims[cand_idx])
    df.insert(0, "job_index", job_index)
    df.insert(1, "job_title", job_row.get("title",""))
    df.insert(2, "job_tax", job_tax)
    df.insert(3, "job_sen", job_sen)
    df.insert(4, "job_skills", [job_skills]*len(df))
    return df.head(top_k)

recommend_candidates_improved(0, top_k=10, retrieve_n=200)




,job_index,job_title,job_tax,job_sen,job_skills,resume_index,Category,semantic_score,final_score,cv_tax,cv_sen,matched_skills,skill_overlap,tax_match,seniority_mismatch
0,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",410,Business Analyst,0.342132,0.339493,data,junior,[(no explicit skill match)],0.000,1.0,0.0
1,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",428,Business Analyst,0.342132,0.339493,data,junior,[(no explicit skill match)],0.000,1.0,0.0
2,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",424,Business Analyst,0.342132,0.339493,data,junior,[(no explicit skill match)],0.000,1.0,0.0
3,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",420,Business Analyst,0.342132,0.339493,data,junior,[(no explicit skill match)],0.000,1.0,0.0
4,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",414,Business Analyst,0.342132,0.339493,data,junior,[(no explicit skill match)],0.000,1.0,0.0
5,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",406,Business Analyst,0.342132,0.339493,data,junior,[(no explicit skill match)],0.000,1.0,0.0
6,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",210,Mechanical Engineer,0.249334,0.299534,data,junior,[technical support],0.125,1.0,0.0
7,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",190,Mechanical Engineer,0.249334,0.299534,data,junior,[technical support],0.125,1.0,0.0
8,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",205,Mechanical Engineer,0.249334,0.299534,data,junior,[technical support],0.125,1.0,0.0
9,0,Support Technologist II 2024-02216,data,junior,"[active directory, desktop support, hardware, ...",185,Mechanical Engineer,0.249334,0.299534,data,junior,[technical support],0.125,1.0,0.0


## Export improved Top-K for a few jobs


### Đánh giá chất lượng gợi ý ứng viên (Candidate Recommendation)

| Tiêu chí | Tệ | Ổn (baseline) | Tốt (improved) | Rất tốt |
|--------|----|---------------|----------------|---------|
| Semantic Score | < 0.30 | 0.30 – 0.40 | **0.40 – 0.55** | > 0.55 |
| Taxonomy Match | 0.0 | — | **1.0** | 1.0 |
| Seniority Mismatch | > 0.5 | 0.2 – 0.5 | **≤ 0.2** | 0.0 |
| Skill Overlap | 0.0 | 0.1 – 0.3 | **≥ 0.3** | ≥ 0.5 |
| Explainability | Không | Một phần | **Đầy đủ** | Rõ ràng |

Trong đó, mô hình improved được đánh giá dựa trên sự kết hợp của nhiều tín hiệu,
không chỉ dựa vào semantic similarity đơn thuần.


In [6]:
job_indices = [0, 10, 50]
dfs = [recommend_candidates_improved(j, top_k=10, retrieve_n=200) for j in job_indices]
df_out = pd.concat(dfs, ignore_index=True)

csv_path = os.path.join(OUT_DIR, "improved_topk.csv")
df_out.to_csv(csv_path, index=False)
csv_path, df_out.head()


('./outputs/candrec_tfidf/improved_topk.csv',
    job_index                           job_title job_tax job_sen  \
 0          0  Support Technologist II 2024-02216    data  junior   
 1          0  Support Technologist II 2024-02216    data  junior   
 2          0  Support Technologist II 2024-02216    data  junior   
 3          0  Support Technologist II 2024-02216    data  junior   
 4          0  Support Technologist II 2024-02216    data  junior   
 
                                           job_skills  resume_index  \
 0  [active directory, desktop support, hardware, ...           410   
 1  [active directory, desktop support, hardware, ...           428   
 2  [active directory, desktop support, hardware, ...           424   
 3  [active directory, desktop support, hardware, ...           420   
 4  [active directory, desktop support, hardware, ...           414   
 
            Category  semantic_score  final_score cv_tax  cv_sen  \
 0  Business Analyst        0.342132     0

## Proxy evaluation metrics (baseline vs improved)
We evaluate improvements without ground-truth by using proxy metrics:
- TaxonomyHit@K: share of Top-K with cv_tax == job_tax
- SeniorityMismatch@K: average mismatch in Top-K
- AvgSkillOverlap@K: average Jaccard overlap
- ExplainabilityRate@K: share of Top-K with >=1 matched skill OR taxonomy match


In [7]:
def recommend_candidates_baseline(job_index: int, top_k: int = 10):
    job_row = jobs.iloc[job_index]
    job_text = build_job_query(job_row)
    q_vec = vectorizer.transform([job_text])
    sims = cosine_similarity(q_vec, X_resumes).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    job_tax = infer_taxonomy(job_text)
    job_sen = infer_seniority(job_text)
    job_skills = extract_skills(job_text)

    js = set(job_skills)
    rows=[]
    for ri, sem in zip(top_idx, sims[top_idx]):
        cv = resumes.iloc[int(ri)]
        cs = set(cv["cv_skills"])
        overlap = len(js & cs) / max(1, len(js | cs))
        tax_match = 1.0 if cv["cv_tax"] == job_tax else 0.0
        mismatch = 0.0
        if job_sen in ["senior","manager"] and cv["cv_sen"] in ["intern","junior"]:
            mismatch = 1.0
        elif job_sen in ["intern","junior"] and cv["cv_sen"] in ["senior","manager"]:
            mismatch = 0.5
        rows.append({"resume_index":int(ri),"semantic_score":float(sem),"cv_tax":cv["cv_tax"],"cv_sen":cv["cv_sen"],
                     "matched_skills":sorted(list(js & cs))[:8],"skill_overlap":overlap,"tax_match":tax_match,
                     "seniority_mismatch":mismatch})
    df=pd.DataFrame(rows)
    return df, {"job_tax":job_tax,"job_sen":job_sen,"job_skills":job_skills}

def compute_metrics(job_index: int, top_k: int = 10, retrieve_n: int = 200):
    base_df, job_meta = recommend_candidates_baseline(job_index, top_k=top_k)
    imp_df = recommend_candidates_improved(job_index, top_k=top_k, retrieve_n=retrieve_n)

    def metrics(df):
        tax_hit = df["tax_match"].mean()
        sen_mis = df["seniority_mismatch"].mean()
        skl = df["skill_overlap"].mean()
        expl = ((df["tax_match"]>0) | (df["matched_skills"].apply(lambda x: len(x)>0))).mean()
        return tax_hit, sen_mis, skl, expl

    b = metrics(base_df)
    i = metrics(imp_df)
    return {
        "job_index": job_index,
        "job_title": jobs.iloc[job_index].get("title",""),
        "TaxonomyHit@K_baseline": b[0],
        "TaxonomyHit@K_improved": i[0],
        "SeniorityMismatch@K_baseline": b[1],
        "SeniorityMismatch@K_improved": i[1],
        "AvgSkillOverlap@K_baseline": b[2],
        "AvgSkillOverlap@K_improved": i[2],
        "ExplainabilityRate@K_baseline": b[3],
        "ExplainabilityRate@K_improved": i[3],
    }

# quick check on a few jobs
pd.DataFrame([compute_metrics(j, top_k=10, retrieve_n=200) for j in [0,10,50]])


,job_index,job_title,TaxonomyHit@K_baseline,TaxonomyHit@K_improved,SeniorityMismatch@K_baseline,SeniorityMismatch@K_improved,AvgSkillOverlap@K_baseline,AvgSkillOverlap@K_improved,ExplainabilityRate@K_baseline,ExplainabilityRate@K_improved
0,0,Support Technologist II 2024-02216,1.0,1.0,0.5,0.0,0.300000,0.050000,1.0,1.0
1,10,IT Field Technician,0.0,0.5,0.0,0.0,0.181818,0.333333,1.0,1.0
2,50,Information Security Analyst,1.0,1.0,0.0,0.0,0.000000,0.000000,1.0,1.0


In [8]:
# run evaluation on a sample of jobs (default 100)
np.random.seed(7)
sample_n = min(100, len(jobs))
job_sample = np.random.choice(len(jobs), size=sample_n, replace=False)

rows = [compute_metrics(int(j), top_k=10, retrieve_n=200) for j in job_sample]
df_rows = pd.DataFrame(rows)

# summary (mean baseline vs improved)
summary = pd.DataFrame({
    "TaxonomyHit@K": [df_rows["TaxonomyHit@K_baseline"].mean(), df_rows["TaxonomyHit@K_improved"].mean()],
    "SeniorityMismatch@K": [df_rows["SeniorityMismatch@K_baseline"].mean(), df_rows["SeniorityMismatch@K_improved"].mean()],
    "AvgSkillOverlap@K": [df_rows["AvgSkillOverlap@K_baseline"].mean(), df_rows["AvgSkillOverlap@K_improved"].mean()],
    "ExplainabilityRate@K": [df_rows["ExplainabilityRate@K_baseline"].mean(), df_rows["ExplainabilityRate@K_improved"].mean()],
}, index=["baseline","improved"])

rows_path = os.path.join(OUT_DIR, "metrics_rows.csv")
summary_path = os.path.join(OUT_DIR, "metrics_summary.csv")
df_rows.to_csv(rows_path, index=False)
summary.to_csv(summary_path)

summary


,TaxonomyHit@K,SeniorityMismatch@K,AvgSkillOverlap@K,ExplainabilityRate@K
baseline,0.684,0.140,0.162832,0.84
improved,0.867,0.008,0.333033,1.00
